In [ ]:
# ============================================================
# CELL 1 — GOOGLE DRIVE MOUNT + FOLDER SETUP + INSTALLATIONS
# Run this cell FIRST
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive
!mkdir -p RSV_Project/data
%cd RSV_Project

# Core bio + ML
!pip install biopython tensorflow scikit-learn seaborn pandas matplotlib

# BERT / Transformers (for DNABERT-style feature extraction)
!pip install transformers torch sentencepiece

# Epitope prediction dependencies
!pip install requests

# Gradio UI
!pip install gradio

print("✅ All installations complete.")


Mounted at /content/drive
/content/drive/MyDrive
/content/drive/MyDrive/RSV_Project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.5 MB/s eta 0:00:00
✅ All installations complete.


In [ ]:
# ============================================================
# CELL 2 — data_preprocessing.py
# UNCHANGED from your original — paste as-is
# ============================================================

%%writefile data_preprocessing.py
from Bio import SeqIO

def load_fasta(file_path, limit=None):
    sequences = []
    for i, record in enumerate(SeqIO.parse(file_path, "fasta")):
        sequences.append(str(record.seq).upper())
        if limit and i >= limit - 1:
            break
    return sequences

def preprocess(sequences):
    print("Initial:", len(sequences))
    sequences = [s for s in sequences if 14000 <= len(s) <= 16000]
    print("After length filter:", len(sequences))
    sequences = [s for s in sequences if s.count('N') / len(s) <= 0.1]
    print("After removing N:", len(sequences))
    sequences = list(set(sequences))
    print("After removing duplicates:", len(sequences))
    return sequences


Overwriting data_preprocessing.py


In [ ]:
# ============================================================
# CELL 3 — redundancy_removal.py
# UNCHANGED from your original
# ============================================================

%%writefile redundancy_removal.py
from collections import Counter
import numpy as np

def get_kmers(seq, k=6):
    return [seq[i:i+k] for i in range(len(seq)-k+1)]

def vectorize(seq, vocab):
    counts = Counter(get_kmers(seq))
    return np.array([counts.get(k, 0) for k in vocab])

def cosine_sim(v1, v2):
    dot = np.dot(v1, v2)
    norm = np.linalg.norm(v1) * np.linalg.norm(v2)
    return dot / norm if norm else 0

def remove_redundant(seqs, threshold=0.995):
    vocab = set()
    for s in seqs:
        vocab.update(get_kmers(s))
    vocab = list(vocab)

    vectors = [vectorize(s, vocab) for s in seqs]

    selected_sequences = []
    selected_vectors   = []
    removed_sequences  = []
    removed_vectors    = []

    for i, v in enumerate(vectors):
        keep = True
        for sv in selected_vectors:
            if cosine_sim(v, sv) > threshold:
                keep = False
                break
        if keep:
            selected_sequences.append(seqs[i])
            selected_vectors.append(v)
        else:
            removed_sequences.append(seqs[i])
            removed_vectors.append(v)

    print("Final sequences after redundancy removal:", len(selected_sequences))
    return selected_sequences, removed_sequences


Overwriting redundancy_removal.py


In [ ]:
# ============================================================
# CELL 4 — bert_feature_extraction.py
# NEW: DNABERT-style k-mer tokenisation + HuggingFace BERT
#      embeddings used as rich feature vectors alongside the
#      existing positional / GC encoding.
# ============================================================

%%writefile bert_feature_extraction.py
"""
DNABERT-style Feature Extraction
---------------------------------
• Splits each DNA sequence into overlapping 6-mers (matching
  the DNABERT tokenisation strategy).
• Loads the pre-trained 'zhihan1996/DNA_bert_6' model from
  HuggingFace (downloads once, ~400 MB, cached in Colab).
• Returns a [CLS] embedding (768-dim) per sequence that can be
  used for downstream classification or as an enriched feature
  alongside the LSTM encoding.

Usage
-----
  from bert_feature_extraction import get_bert_embeddings
  embeddings = get_bert_embeddings(["ATGCGT...", ...], batch_size=8)
  # → numpy array of shape (N, 768)
"""

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

# ── model id ──────────────────────────────────────────────────
DNABERT_MODEL = "zhihan1996/DNA_bert_6"

# ── lazy globals (loaded once) ────────────────────────────────
_tokenizer = None
_model     = None


def _load_model():
    global _tokenizer, _model
    if _tokenizer is None:
        print("Loading DNABERT-6 … (first run downloads ~400 MB)")
        _tokenizer = AutoTokenizer.from_pretrained(
            DNABERT_MODEL, trust_remote_code=True
        )
        _model = AutoModel.from_pretrained(
            DNABERT_MODEL, trust_remote_code=True
        )
        _model.eval()
        if torch.cuda.is_available():
            _model = _model.cuda()
        print("✅ DNABERT-6 loaded.")


# ── helpers ───────────────────────────────────────────────────
def _seq_to_kmers(seq: str, k: int = 6) -> str:
    """'ATGCGT' → 'ATG TGC GCG CGT'  (space-separated 6-mers)."""
    kmers = [seq[i:i+k] for i in range(len(seq) - k + 1)]
    return " ".join(kmers)


def _truncate(seq: str, max_len: int = 512, k: int = 6) -> str:
    """
    BERT has a 512-token limit.
    Each k-mer is one token; keep the first (512-2) k-mers
    (−2 for [CLS] and [SEP]).
    """
    kmers = [seq[i:i+k] for i in range(len(seq) - k + 1)]
    kmers = kmers[: max_len - 2]
    return " ".join(kmers)


# ── public API ────────────────────────────────────────────────
def get_bert_embeddings(
    sequences: list,
    batch_size: int = 8,
    k: int = 6
) -> np.ndarray:
    """
    Parameters
    ----------
    sequences  : list of uppercase DNA strings
    batch_size : sequences per forward pass (reduce if OOM)
    k          : k-mer length (must match loaded model, default 6)

    Returns
    -------
    numpy ndarray of shape (len(sequences), 768)
    """
    _load_model()

    device     = "cuda" if torch.cuda.is_available() else "cpu"
    all_embeds = []

    for start in range(0, len(sequences), batch_size):
        batch = sequences[start : start + batch_size]
        kmer_seqs = [_truncate(s, k=k) for s in batch]

        enc = _tokenizer(
            kmer_seqs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        )
        enc = {key: val.to(device) for key, val in enc.items()}

        with torch.no_grad():
            out = _model(**enc)

        # [CLS] token embedding → shape (batch, 768)
        cls_embed = out.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeds.append(cls_embed)

        if (start // batch_size) % 5 == 0:
            print(
                f"  BERT embeddings: {min(start+batch_size, len(sequences))}"
                f"/{len(sequences)}"
            )

    return np.vstack(all_embeds)   # (N, 768)


# ── per-window BERT features for the LSTM dataset ─────────────
def bert_window_features(
    seq: str,
    window: int = 50,
    stride: int = 10,
    k: int = 6
) -> np.ndarray:
    """
    Extract BERT CLS embeddings for sliding windows of a sequence.
    Useful to create richer per-position features alongside the
    existing one-hot + GC + position encoding.

    Returns array of shape (num_windows, 768).
    """
    _load_model()

    device  = "cuda" if torch.cuda.is_available() else "cpu"
    windows = []

    for i in range(0, len(seq) - window, stride):
        windows.append(seq[i:i+window])

    embeddings = []
    for start in range(0, len(windows), 16):   # batch=16
        batch     = windows[start : start + 16]
        kmer_seqs = [_truncate(s, k=k) for s in batch]
        enc       = _tokenizer(
            kmer_seqs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        )
        enc = {key: val.to(device) for key, val in enc.items()}
        with torch.no_grad():
            out = _model(**enc)
        embeddings.append(out.last_hidden_state[:, 0, :].cpu().numpy())

    return np.vstack(embeddings) if embeddings else np.array([])


Overwriting bert_feature_extraction.py


In [ ]:
# ============================================================
# CELL 5 — feature_extraction.py  (UPDATED)
# Original one-hot + position + GC encoding is kept intact.
# Added: optional BERT embedding concatenation per window.
# ============================================================

%%writefile feature_extraction.py
import numpy as np

# ── one-hot mapping (A T G C) ─────────────────────────────────
mapping = {
    'A': [1, 0, 0, 0],
    'T': [0, 1, 0, 0],
    'G': [0, 0, 1, 0],
    'C': [0, 0, 0, 1],
}

# ── RSV gene coordinate map (NCBI RefSeq NC_001781 / your GB) ─
RSV_GENE_MAP = {
    "NS1":  (99,    518),
    "NS2":  (628,   1002),
    "N":    (1140,  2315),
    "P":    (2347,  3072),
    "M":    (3255,  4025),
    "SH":   (4295,  4489),
    "G":    (4681,  5646),
    "F":    (5726,  7450),
    "M2-1": (7669,  8253),
    "M2-2": (8228,  8494),
    "L":    (8561,  15058),
}


def get_gene_label(position: int) -> str:
    """Return the RSV gene name for a genome position (0-based)."""
    for gene, (start, end) in RSV_GENE_MAP.items():
        if start <= position <= end:
            return gene
    return "Intergenic"


def encode_sequence(seq: str) -> np.ndarray:
    """
    Per-nucleotide feature vector (6-dim):
        [one_hot×4, relative_position, local_GC_content]
    """
    enc = []
    for i, b in enumerate(seq):
        if b not in mapping:
            continue
        pos = i / len(seq)
        gc  = (
            seq[max(0, i-25):i+25].count('G') +
            seq[max(0, i-25):i+25].count('C')
        ) / 50
        enc.append(mapping[b] + [pos] + [gc])
    return np.array(enc)   # shape (L, 6)


def create_dataset(
    seqs,
    window: int = 50,
    max_samples: int = 300000
):
    """
    Sliding-window dataset for next-nucleotide prediction.
    Returns X (N, window, 6) and y (N, 4).
    """
    X, y = [], []
    c = 0
    for s in seqs:
        e = encode_sequence(s)
        for i in range(len(e) - window):
            X.append(e[i:i+window])
            y.append(e[i+window][:4])
            c += 1
            if c >= max_samples:
                return np.array(X), np.array(y)
    return np.array(X), np.array(y)


Overwriting feature_extraction.py


In [ ]:
# ============================================================
# CELL 6 — model.py  (UNCHANGED from your original)
# ============================================================

%%writefile model.py
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import random, tensorflow as tf, numpy as np

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

def plot_training(history):
    plt.figure()
    plt.plot(history.history['accuracy'],     label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.xlabel('Epochs'); plt.ylabel('Accuracy')
    plt.title('Accuracy vs Epochs'); plt.legend(); plt.grid()
    plt.savefig("/content/drive/MyDrive/RSV_Project/accuracy_plot.png")
    plt.show()

    plt.figure()
    plt.plot(history.history['loss'],     label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.xlabel('Epochs'); plt.ylabel('Loss')
    plt.title('Loss vs Epochs'); plt.legend(); plt.grid()
    plt.savefig("/content/drive/MyDrive/RSV_Project/loss_plot.png")
    plt.show()

def build_model(shape):
    m = Sequential()
    m.add(Input(shape=shape))
    m.add(LSTM(256, return_sequences=True))
    m.add(Dropout(0.4))
    m.add(LSTM(128))
    m.add(Dropout(0.3))
    m.add(Dense(128, activation='relu'))
    m.add(Dense(64,  activation='relu'))
    m.add(Dense(4,   activation='softmax'))
    m.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )
    return m

def train_model(X, y):
    m = build_model((X.shape[1], X.shape[2]))
    early_stop = EarlyStopping(
        monitor='val_loss', patience=3, restore_best_weights=True
    )
    reduce_lr = ReduceLROnPlateau(factor=0.5, patience=2)
    history = m.fit(
        X, y,
        epochs=25,
        batch_size=128,
        validation_split=0.1,
        callbacks=[early_stop, reduce_lr]
    )
    plot_training(history)
    print("\nFINAL ACCURACY:")
    print("Train:", history.history['accuracy'][-1])
    print("Val:",   history.history['val_accuracy'][-1])
    return m


Overwriting model.py


In [ ]:
# ============================================================
# CELL 8 — evaluation.py  (UNCHANGED)
# ============================================================

%%writefile evaluation.py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

def evaluate_model(model, X, y):
    pred   = model.predict(X, verbose=0)
    y_true = np.argmax(y,    axis=1)
    y_pred = np.argmax(pred, axis=1)

    precision = precision_score(y_true, y_pred, average='weighted')
    recall    = recall_score   (y_true, y_pred, average='weighted')
    f1        = f1_score       (y_true, y_pred, average='weighted')

    print("\n===== EVALUATION METRICS =====")
    print("Precision:", precision)
    print("Recall:",    recall)
    print("F1-score:",  f1)

    print("\n===== CLASSIFICATION REPORT =====")
    print(classification_report(y_true, y_pred, target_names=['A','T','G','C']))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['A','T','G','C'],
                yticklabels=['A','T','G','C'])
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.savefig("/content/drive/MyDrive/RSV_Project/confusion_matrix.png")
    plt.show()


Overwriting evaluation.py


In [ ]:
# ============================================================
# CELL 9 — mutation_prediction.py  (UPDATED)
# Added: gene-level labelling for every predicted mutation
#        using the RSV_GENE_MAP from feature_extraction.
# ============================================================

%%writefile mutation_prediction.py
import numpy as np
from tensorflow.keras.models import load_model
from feature_extraction import encode_sequence, get_gene_label

def load_model_file():
    return load_model("model.keras")

def predict_mutations(
    model,
    seq: str,
    window: int    = 50,
    max_pos: int   = 2000,
    threshold: float = 0.6
) -> list:
    """
    Slide a window across `seq`, predict the next nucleotide,
    and flag positions where prediction ≠ actual with
    confidence > threshold.

    Each returned dict now includes:
        position, actual, predicted, confidence, gene
    """
    enc      = encode_sequence(seq)
    base_map = ['A', 'T', 'G', 'C']

    X         = []
    positions = []

    for i in range(min(len(enc) - window, max_pos)):
        X.append(enc[i:i+window])
        positions.append(i)

    X    = np.array(X)
    preds = model.predict(X, verbose=0)

    muts = []
    for idx, p in enumerate(preds):
        pred_base   = base_map[np.argmax(p)]
        confidence  = float(np.max(p))
        actual_base = seq[positions[idx] + window]

        if pred_base != actual_base and confidence > threshold:
            pos = positions[idx] + window
            muts.append({
                "position":   pos,
                "actual":     actual_base,
                "predicted":  pred_base,
                "confidence": confidence,
                "gene":       get_gene_label(pos),   # ← NEW
            })

    return muts


Overwriting mutation_prediction.py


In [ ]:
# ============================================================
# CELL 10 — protein_analysis.py  (UPDATED)
# Added: per-gene protein extraction for ALL 11 RSV genes
#        (NS1, NS2, N, P, M, SH, G, F, M2-1, M2-2, L)
# ============================================================

%%writefile protein_analysis.py
from Bio.Seq import Seq

# ── RSV gene coordinate map (same as feature_extraction.py) ──
RSV_GENE_MAP = {
    "NS1":  (99,    518),
    "NS2":  (628,   1002),
    "N":    (1140,  2315),
    "P":    (2347,  3072),
    "M":    (3255,  4025),
    "SH":   (4295,  4489),
    "G":    (4681,  5646),
    "F":    (5726,  7450),
    "M2-1": (7669,  8253),
    "M2-2": (8228,  8494),
    "L":    (8561,  15058),
}


# ── helpers ───────────────────────────────────────────────────
def dna_to_protein(seq: str) -> str:
    """Translate the full sequence in frame (truncated to codon boundary)."""
    seq = seq[:len(seq) // 3 * 3]
    return str(Seq(seq).translate())


def extract_gene_sequence(full_seq: str, gene: str) -> str:
    """
    Extract the coding sequence for a named RSV gene.
    Coordinates are 1-based (GenBank style) → converted to 0-based.
    """
    if gene not in RSV_GENE_MAP:
        raise ValueError(f"Unknown gene: {gene}. "
                         f"Valid genes: {list(RSV_GENE_MAP.keys())}")
    start, end = RSV_GENE_MAP[gene]
    return full_seq[start - 1 : end]   # end is inclusive in GB


def get_all_gene_proteins(full_seq: str) -> dict:
    """
    Translate every RSV gene from the full-genome sequence.
    Returns  { gene_name: protein_string }
    """
    proteins = {}
    for gene in RSV_GENE_MAP:
        cds = extract_gene_sequence(full_seq, gene)
        proteins[gene] = dna_to_protein(cds)
    return proteins


def apply_mutations(seq: str, muts: list) -> str:
    """Apply predicted mutations to a sequence string."""
    seq = list(seq)
    for m in muts:
        pos = m["position"]
        if pos < len(seq):
            seq[pos] = m["predicted"]
    return "".join(seq)


def compare_proteins(p1: str, p2: str) -> tuple:
    """
    Compare two protein strings residue by residue.
    Returns (normal_changes, stop_codon_changes).
    """
    protein_changes    = []
    stop_codon_changes = []

    for i in range(min(len(p1), len(p2))):
        if p1[i] == p2[i]:
            continue
        change = {"position": i, "wild_aa": p1[i], "mutated_aa": p2[i]}
        if p1[i] == '*' or p2[i] == '*':
            stop_codon_changes.append(change)
        else:
            protein_changes.append(change)

    return protein_changes, stop_codon_changes


# ── per-gene protein comparison ───────────────────────────────
def compare_gene_proteins(
    wild_seq: str,
    mut_seq:  str
) -> dict:
    """
    For every RSV gene, compare the wild-type protein against the
    mutated protein and return a dict of results.

    Returns
    -------
    {
      gene_name: {
        "wild_protein":  str,
        "mut_protein":   str,
        "aa_changes":    list of dicts,
        "stop_changes":  list of dicts,
      },
      ...
    }
    """
    results = {}
    for gene in RSV_GENE_MAP:
        try:
            wild_cds = extract_gene_sequence(wild_seq, gene)
            mut_cds  = extract_gene_sequence(mut_seq,  gene)

            wp = dna_to_protein(wild_cds)
            mp = dna_to_protein(mut_cds)

            aa_changes, stop_changes = compare_proteins(wp, mp)

            results[gene] = {
                "wild_protein": wp,
                "mut_protein":  mp,
                "aa_changes":   aa_changes,
                "stop_changes": stop_changes,
            }
        except Exception as e:
            results[gene] = {"error": str(e)}

    return results


Overwriting protein_analysis.py


In [ ]:
# ============================================================
# CELL 11 — ddg_prediction.py  (UPDATED — FULL RULE-BASED)
# Expanded biochemical property groups and detailed rules
# covering ALL major amino-acid interaction classes:
#   hydrophobic, hydrophilic/polar, charged (pos/neg),
#   aromatic, small/flexible, helix-breakers, cysteines.
# ============================================================

%%writefile ddg_prediction.py
"""
Rule-Based ΔΔG Estimation
--------------------------
Each substitution is classified by the biochemical property
groups of the wild-type and mutant residues.

ΔΔG > 0  → destabilising  (positive = unfavourable)
ΔΔG < 0  → stabilising    (negative = favourable)
ΔΔG = 0  → neutral / conservative
"""

# ── amino-acid property groups ────────────────────────────────
HYDROPHOBIC_CORE   = set("AVLIMFW")     # buried-core hydrophobics
HYDROPHOBIC_MILD   = set("P")           # weakly hydrophobic
POLAR_UNCHARGED    = set("STNQYC")      # polar but neutral
CHARGED_POS        = set("KRH")         # positively charged
CHARGED_NEG        = set("DE")          # negatively charged
AROMATIC           = set("FYW")         # aromatic side-chains
SMALL_FLEXIBLE     = set("GA")          # glycine / alanine
CYSTEINE           = set("C")           # potential disulfide
HELIX_BREAKERS     = set("P")           # proline breaks α-helix

# convenience unions
ALL_CHARGED        = CHARGED_POS | CHARGED_NEG
ALL_HYDROPHOBIC    = HYDROPHOBIC_CORE | HYDROPHOBIC_MILD


# ── scoring matrix ────────────────────────────────────────────
def classify_change(w: str, m: str) -> float:
    """
    Return a ΔΔG estimate (kcal/mol proxy) for a single
    wild-type → mutant amino-acid substitution.
    """
    # ── special cases ─────────────────────────────────────────
    if w == '*' or m == '*':
        return 3.0          # stop codon gain/loss: severe

    if w == m:
        return 0.0          # synonymous

    # ── disulfide-forming / breaking ──────────────────────────
    if w in CYSTEINE and m not in CYSTEINE:
        return 2.0          # loss of potential disulfide

    if m in CYSTEINE and w not in CYSTEINE:
        return -0.5         # gain of potential disulfide (can be good)

    # ── proline: helix-breaker introduced / removed ───────────
    if m in HELIX_BREAKERS and w not in HELIX_BREAKERS:
        return 1.8          # proline in helix → destabilising

    if w in HELIX_BREAKERS and m not in HELIX_BREAKERS:
        return -0.5         # proline removed → slightly stabilising

    # ── glycine: adds flexibility ─────────────────────────────
    if m == 'G' and w != 'G':
        return 1.0          # loss of side-chain constraint

    if w == 'G' and m != 'G':
        return -0.3         # gain of side-chain → slight stabilisation

    # ── hydrophobic core disruption ───────────────────────────
    if w in HYDROPHOBIC_CORE and m in ALL_CHARGED:
        return 2.5          # burying a charge → very destabilising

    if w in HYDROPHOBIC_CORE and m in POLAR_UNCHARGED:
        return 1.5          # buried polar → destabilising

    if w in HYDROPHOBIC_CORE and m in HYDROPHOBIC_CORE:
        # conservative hydrophobic swap – small effect
        return 0.2

    # ── polar to hydrophobic ──────────────────────────────────
    if w in POLAR_UNCHARGED and m in HYDROPHOBIC_CORE:
        return -1.0         # improved hydrophobic packing

    # ── charge neutralisation ─────────────────────────────────
    if w in ALL_CHARGED and m in POLAR_UNCHARGED:
        return 0.8          # charge removed, mild destabilisation

    if w in ALL_CHARGED and m in HYDROPHOBIC_CORE:
        return 1.5          # charge buried → destabilising

    # ── charge–charge swaps ───────────────────────────────────
    if w in CHARGED_POS and m in CHARGED_NEG:
        return 2.0          # sign flip → severe electrostatic clash

    if w in CHARGED_NEG and m in CHARGED_POS:
        return 2.0

    if w in CHARGED_POS and m in CHARGED_POS:
        return 0.4          # same sign, different residue

    if w in CHARGED_NEG and m in CHARGED_NEG:
        return 0.4

    # ── aromatic stacking ─────────────────────────────────────
    if w in AROMATIC and m not in AROMATIC:
        return 1.2          # loss of aromatic stacking

    if m in AROMATIC and w not in AROMATIC:
        return -0.4         # gain of aromatic interaction

    # ── default small / medium effect ────────────────────────
    return 0.3


# ── interpret a single ΔΔG value ─────────────────────────────
def interpret_ddg(val: float) -> str:
    if val >= 2.0:
        return "Highly Destabilising"
    elif val >= 1.0:
        return "Moderately Destabilising"
    elif val > 0:
        return "Mildly Destabilising"
    elif val == 0:
        return "Neutral"
    elif val >= -0.5:
        return "Slightly Stabilising"
    else:
        return "Stabilising"


# ── public API ────────────────────────────────────────────────
def compute_ddg(protein_changes: list) -> tuple:
    """
    Parameters
    ----------
    protein_changes : list of dicts with keys 'position',
                      'wild_aa', 'mutated_aa'

    Returns
    -------
    (results_list, total_ddg)
        results_list : list of dicts – each with
                       position, wild, mutated, ddg, interpretation
        total_ddg    : float – cumulative ΔΔG
    """
    results   = []
    total_ddg = 0.0

    for change in protein_changes:
        w   = change["wild_aa"]
        m   = change["mutated_aa"]
        ddg = classify_change(w, m)
        total_ddg += ddg

        results.append({
            "position":       change["position"],
            "wild":           w,
            "mutated":        m,
            "ddg":            round(ddg, 3),
            "interpretation": interpret_ddg(ddg),
        })

    return results, round(total_ddg, 3)


# ── stability suggestions ─────────────────────────────────────
def get_stability_suggestions(ddg_results: list) -> list:
    """
    For every highly destabilising mutation (ΔΔG ≥ 1.0),
    suggest a back-substitution or stabilising replacement
    with biochemical rationale.
    """
    suggestions = []

    for d in ddg_results:
        if d["ddg"] < 1.0:
            continue                      # only flag significant hits

        current = d["mutated"]

        # ── rule-based suggestion ─────────────────────────────
        if current in CHARGED_POS:
            suggested = "L"
            reason    = (
                "Replace positively charged residue with Leucine "
                "(hydrophobic) to restore core packing stability."
            )
        elif current in CHARGED_NEG:
            suggested = "V"
            reason    = (
                "Replace negatively charged residue with Valine "
                "to reduce electrostatic disruption in hydrophobic core."
            )
        elif current in POLAR_UNCHARGED:
            suggested = "V"
            reason    = (
                "Replace polar residue with Valine to reduce "
                "solvent exposure and improve hydrophobic folding."
            )
        elif current == 'G':
            suggested = "A"
            reason    = (
                "Alanine provides a methyl side-chain, reducing "
                "excessive flexibility introduced by Glycine."
            )
        elif current == 'P':
            suggested = "A"
            reason    = (
                "Alanine replaces helix-breaking Proline, "
                "restoring secondary structure continuity."
            )
        elif current in AROMATIC:
            suggested = "L"
            reason    = (
                "Leucine can approximate aromatic burial in the "
                "hydrophobic core when the ring system is disruptive."
            )
        else:
            suggested = "L"
            reason    = (
                "Leucine is a conservative hydrophobic substitute "
                "for general core stabilisation."
            )

        suggestions.append({
            "Position":     d["position"],
            "Wild AA":      d["wild"],
            "Current Mut":  current,
            "Suggested AA": suggested,
            "ΔΔG":          d["ddg"],
            "Reason":       reason,
        })

    return suggestions


Overwriting ddg_prediction.py


In [ ]:
%%writefile epitope_prediction.py
"""
Epitope Prediction Module
--------------------------
Methods
-------
1. B-cell (linear)
   Parker hydrophilicity scale  (sliding window = 7)
   Kolaskar-Tongaonkar antigenicity (sliding window = 7)

2. T-cell MHC-I
   IEDB Analysis Resource REST API (updated endpoint)
   Method: netmhcpan  |  Allele: HLA-A*02:01
   Peptide length: 9-mer

3. T-cell MHC-II
   IEDB REST, method: netmhciipan  |  Allele: DRB1*01:01
   Peptide length: 15-mer
"""

import numpy as np
import requests
from protein_analysis import (
    extract_gene_sequence,
    dna_to_protein,
    RSV_GENE_MAP,
)

# ══════════════════════════════════════════════════════════════
# 1. B-CELL EPITOPE PREDICTION (Rule-Based — works offline)
# ══════════════════════════════════════════════════════════════

PARKER_SCALE = {
    'A': -0.5, 'R':  3.0, 'N':  0.2, 'D':  3.0, 'C': -1.0,
    'Q':  0.2, 'E':  3.0, 'G':  0.0, 'H': -0.5, 'I': -1.8,
    'L': -1.8, 'K':  3.0, 'M': -1.3, 'F': -2.5, 'P':  0.0,
    'S':  0.3, 'T': -0.4, 'W': -3.4, 'Y': -2.3, 'V': -1.5,
}

KOLASKAR_SCALE = {
    'A': 1.064, 'R': 0.873, 'N': 0.776, 'D': 0.908, 'C': 1.412,
    'Q': 0.853, 'E': 0.932, 'G': 0.874, 'H': 1.105, 'I': 1.152,
    'L': 1.250, 'K': 0.930, 'M': 0.826, 'F': 1.091, 'P': 1.064,
    'S': 1.012, 'T': 0.909, 'W': 0.893, 'Y': 1.161, 'V': 1.383,
}

def _sliding_window_score(protein, scale, window=7):
    scores = []
    for i in range(len(protein) - window + 1):
        segment = protein[i:i+window]
        vals = [scale.get(aa, 0.0) for aa in segment]
        scores.append((i, i + window, round(np.mean(vals), 4)))
    return scores

def _filter_top_windows(windows, percentile=75):
    if not windows:
        return []
    scores = [w[2] for w in windows]
    cutoff = np.percentile(scores, percentile)
    return [w for w in windows if w[2] >= cutoff]

def predict_bcell_epitopes(protein, window=7, percentile=75):
    protein = protein.replace('*', '')
    parker_raw   = _sliding_window_score(protein, PARKER_SCALE,   window)
    kolaskar_raw = _sliding_window_score(protein, KOLASKAR_SCALE, window)
    parker_top   = _filter_top_windows(parker_raw,   percentile)
    kolaskar_top = _filter_top_windows(kolaskar_raw, percentile)

    def _fmt(wins, method):
        return [
            {"start": s, "end": e, "score": sc,
             "peptide": protein[s:e], "method": method}
            for s, e, sc in wins
        ]
    return {
        "parker":   _fmt(parker_top,   "Parker"),
        "kolaskar": _fmt(kolaskar_top, "Kolaskar-Tongaonkar"),
    }

# ══════════════════════════════════════════════════════════════
# 2. T-CELL MHC-I PREDICTION  (IEDB updated endpoint + format)
# ══════════════════════════════════════════════════════════════

# Updated IEDB API URLs (2024 format)
IEDB_MHC1_URL  = "https://tools.iedb.org/tools_api/mhci/"
IEDB_MHC2_URL  = "https://tools.iedb.org/tools_api/mhcii/"

def _chunk_protein(protein, chunk_size=200):
    protein = protein.replace('*', '')
    return [
        protein[i:i+chunk_size]
        for i in range(0, len(protein), chunk_size)
        if len(protein[i:i+chunk_size]) >= 9
    ]

def predict_mhci_epitopes(
    protein,
    allele="HLA-A*02:01",
    length=9,
    method="netmhcpan",
    score_threshold=500.0,
):
    protein = protein.replace('*', '')
    chunks  = _chunk_protein(protein)
    results = []
    offset  = 0

    for chunk in chunks:
        # IEDB updated format requires JSON body
        payload = {
            "method":        method,
            "sequence_text": chunk,
            "allele":        allele,
            "length":        str(length),
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}

        try:
            resp = requests.post(
                IEDB_MHC1_URL,
                data=payload,
                headers=headers,
                timeout=45,
            )

            if resp.status_code == 405:
                # Try GET fallback with params
                resp = requests.get(
                    IEDB_MHC1_URL,
                    params=payload,
                    timeout=45,
                )

            if resp.status_code not in (200, 201):
                print(f"  ⚠ IEDB MHC-I returned {resp.status_code} — using rule-based fallback")
                results.extend(_rule_based_mhci(chunk, offset, allele, length, score_threshold))
                offset += len(chunk)
                continue

            lines = resp.text.strip().split('\n')
            for line in lines[1:]:
                parts = line.split('\t')
                if len(parts) < 7:
                    continue
                try:
                    start   = int(parts[2]) - 1 + offset
                    end     = start + length
                    peptide = parts[5]
                    ic50    = float(parts[7]) if len(parts) > 7 else 9999.0
                    results.append({
                        "start":   start,
                        "end":     end,
                        "peptide": peptide,
                        "allele":  allele,
                        "ic50_nM": round(ic50, 2),
                        "binder":  "Yes" if ic50 <= score_threshold else "No",
                        "type":    "MHC-I",
                    })
                except (ValueError, IndexError):
                    continue

        except requests.exceptions.RequestException as e:
            print(f"  ⚠ IEDB MHC-I unreachable ({e}) — using rule-based fallback")
            results.extend(_rule_based_mhci(chunk, offset, allele, length, score_threshold))

        offset += len(chunk)

    return [r for r in results if r["binder"] == "Yes"]


def predict_mhcii_epitopes(
    protein,
    allele="DRB1*01:01",
    length=15,
    method="netmhciipan",
    score_threshold=1.0,
):
    protein = protein.replace('*', '')
    chunks  = _chunk_protein(protein, chunk_size=300)
    results = []
    offset  = 0

    for chunk in chunks:
        payload = {
            "method":        method,
            "sequence_text": chunk,
            "allele":        allele,
            "length":        str(length),
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}

        try:
            resp = requests.post(
                IEDB_MHC2_URL,
                data=payload,
                headers=headers,
                timeout=45,
            )

            if resp.status_code == 405:
                resp = requests.get(
                    IEDB_MHC2_URL,
                    params=payload,
                    timeout=45,
                )

            if resp.status_code not in (200, 201):
                print(f"  ⚠ IEDB MHC-II returned {resp.status_code} — using rule-based fallback")
                results.extend(_rule_based_mhcii(chunk, offset, allele, length))
                offset += len(chunk)
                continue

            lines = resp.text.strip().split('\n')
            for line in lines[1:]:
                parts = line.split('\t')
                if len(parts) < 6:
                    continue
                try:
                    start   = int(parts[2]) - 1 + offset
                    end     = start + length
                    peptide = parts[5]
                    rank    = float(parts[8]) if len(parts) > 8 else 99.0
                    results.append({
                        "start":      start,
                        "end":        end,
                        "peptide":    peptide,
                        "allele":     allele,
                        "percentile": round(rank, 3),
                        "binder":     "Yes" if rank <= score_threshold else "No",
                        "type":       "MHC-II",
                    })
                except (ValueError, IndexError):
                    continue

        except requests.exceptions.RequestException as e:
            print(f"  ⚠ IEDB MHC-II unreachable ({e}) — using rule-based fallback")
            results.extend(_rule_based_mhcii(chunk, offset, allele, length))

        offset += len(chunk)

    return [r for r in results if r["binder"] == "Yes"]


# ══════════════════════════════════════════════════════════════
# 3. RULE-BASED MHC FALLBACK
#    When IEDB API is unavailable, use physico-chemical rules
#    to predict likely binders. Based on anchor residue rules
#    for HLA-A*02:01 (most studied allele).
# ══════════════════════════════════════════════════════════════

# HLA-A*02:01 anchor positions: pos 2 = L/M/V, pos 9 = L/V/I
HLA_A0201_ANCHOR_P2  = set("LMV")
HLA_A0201_ANCHOR_P9  = set("LVI")

# Hydrophobic amino acids (preferred in MHC groove)
HYDROPHOBIC_AA = set("AVLIMFYW")

def _rule_based_mhci(
    protein: str,
    offset: int,
    allele: str,
    length: int = 9,
    score_threshold: float = 500.0,
) -> list:
    """
    Rule-based MHC-I 9-mer prediction using HLA-A*02:01
    anchor residue rules:
      - Position 2 (index 1): L, M, or V  (primary anchor)
      - Position 9 (index 8): L, V, or I  (primary anchor)
      - Higher hydrophobicity score → stronger predicted binder
    """
    results = []
    protein = protein.replace('*', '')

    for i in range(len(protein) - length + 1):
        peptide = protein[i:i+length]

        if len(peptide) < length:
            continue

        # Anchor residue check
        anchor2_ok = peptide[1] in HLA_A0201_ANCHOR_P2
        anchor9_ok = peptide[length-1] in HLA_A0201_ANCHOR_P9

        # Hydrophobicity score (proxy for binding affinity)
        hydro_score = sum(
            1 for aa in peptide if aa in HYDROPHOBIC_AA
        ) / length

        # Scoring: both anchors + good hydrophobicity = strong binder
        if anchor2_ok and anchor9_ok:
            predicted_ic50 = 50.0 + (1 - hydro_score) * 200   # ~50–250 nM
            binder = "Yes"
        elif anchor2_ok or anchor9_ok:
            predicted_ic50 = 300.0 + (1 - hydro_score) * 200  # ~300–500 nM
            binder = "Yes" if predicted_ic50 <= score_threshold else "No"
        else:
            continue   # skip non-binders

        if binder == "Yes":
            results.append({
                "start":   i + offset,
                "end":     i + offset + length,
                "peptide": peptide,
                "allele":  allele,
                "ic50_nM": round(predicted_ic50, 2),
                "binder":  "Yes",
                "type":    "MHC-I (rule-based)",
            })

    return results


# DRB1*01:01 prefers hydrophilic/charged residues at P1 anchor
DRB1_ANCHOR_P1 = set("YFWLIV")   # P1 anchor: large hydrophobic

def _rule_based_mhcii(
    protein: str,
    offset: int,
    allele: str,
    length: int = 15,
) -> list:
    """
    Rule-based MHC-II 15-mer prediction using DRB1*01:01
    P1 anchor residue rule:
      - Position 1 (index 0): Y, F, W, L, I, V preferred
    """
    results = []
    protein = protein.replace('*', '')

    for i in range(len(protein) - length + 1):
        peptide = protein[i:i+length]
        if len(peptide) < length:
            continue

        # P1 anchor
        if peptide[0] not in DRB1_ANCHOR_P1:
            continue

        # Overall charge score (MHC-II prefers amphipathic peptides)
        charged = sum(1 for aa in peptide if aa in "DEKRH")
        hydro   = sum(1 for aa in peptide if aa in HYDROPHOBIC_AA)

        # Simple amphipathicity check
        if hydro >= 4 and charged >= 2:
            predicted_rank = 0.5   # strong binder
        elif hydro >= 3:
            predicted_rank = 0.9
        else:
            continue

        results.append({
            "start":      i + offset,
            "end":        i + offset + length,
            "peptide":    peptide,
            "allele":     allele,
            "percentile": predicted_rank,
            "binder":     "Yes",
            "type":       "MHC-II (rule-based)",
        })

    return results


# ══════════════════════════════════════════════════════════════
# 4. PER-GENE EPITOPE PROFILING
# ══════════════════════════════════════════════════════════════

def run_full_epitope_analysis(full_genome_seq, run_mhc=True):
    analysis = {}

    for gene in RSV_GENE_MAP:
        print(f"\n── Epitope analysis: {gene} ──")
        try:
            cds     = extract_gene_sequence(full_genome_seq, gene)
            protein = dna_to_protein(cds).replace('*', '')

            bcell = predict_bcell_epitopes(protein)

            mhci_res  = []
            mhcii_res = []

            if run_mhc and len(protein) >= 9:
                print(f"   MHC-I  ({gene}) …")
                mhci_res  = predict_mhci_epitopes(protein)
                print(f"   MHC-II ({gene}) …")
                mhcii_res = predict_mhcii_epitopes(protein)

            analysis[gene] = {
                "protein":        protein,
                "length":         len(protein),
                "bcell_parker":   bcell["parker"],
                "bcell_kolaskar": bcell["kolaskar"],
                "mhci":           mhci_res,
                "mhcii":          mhcii_res,
            }
            print(
                f"   B-cell Parker={len(bcell['parker'])} "
                f"Kolaskar={len(bcell['kolaskar'])}  "
                f"MHC-I={len(mhci_res)}  MHC-II={len(mhcii_res)}"
            )

        except Exception as e:
            print(f"   ⚠ Error for {gene}: {e}")
            analysis[gene] = {"error": str(e)}

    return analysis


# ══════════════════════════════════════════════════════════════
# 5. MUTATION IMPACT ON EPITOPES
# ══════════════════════════════════════════════════════════════

def check_mutation_in_epitope(mutations, epitope_analysis):
    flagged = []

    for mut in mutations:
        pos  = mut["position"]
        gene = mut.get("gene", "Unknown")
        overlapping = []

        if gene in epitope_analysis and "error" not in epitope_analysis[gene]:
            data       = epitope_analysis[gene]
            gene_start = RSV_GENE_MAP.get(gene, (0, 0))[0] - 1
            prot_pos   = (pos - gene_start) // 3

            for src in ["bcell_parker", "bcell_kolaskar", "mhci", "mhcii"]:
                for ep in data.get(src, []):
                    if ep["start"] <= prot_pos <= ep["end"]:
                        overlapping.append({
                            "type":    src,
                            "peptide": ep.get("peptide", ""),
                            "start":   ep["start"],
                            "end":     ep["end"],
                        })

        if overlapping:
            flagged.append({
                "position":             pos,
                "gene":                 gene,
                "actual":               mut["actual"],
                "predicted":            mut["predicted"],
                "confidence":           mut["confidence"],
                "overlapping_epitopes": overlapping,
                "epitope_disruption":   "⚠ HIGH IMPACT",
            })

    return flagged

Overwriting epitope_prediction.py


In [ ]:
# ============================================================
# CELL 13 — visualization.py  (UPDATED)
# Added: gene-level mutation bar chart, epitope map,
#        per-gene ΔΔG heatmap
# ============================================================

%%writefile visualization.py
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter

BASE_PATH = "/content/drive/MyDrive/RSV_Project/"

# ── RSV gene map for genome-annotation strip ─────────────────
RSV_GENE_MAP = {
    "NS1":  (99,    518),
    "NS2":  (628,   1002),
    "N":    (1140,  2315),
    "P":    (2347,  3072),
    "M":    (3255,  4025),
    "SH":   (4295,  4489),
    "G":    (4681,  5646),
    "F":    (5726,  7450),
    "M2-1": (7669,  8253),
    "M2-2": (8228,  8494),
    "L":    (8561,  15058),
}

GENE_COLORS = [
    "#e6194b","#3cb44b","#ffe119","#4363d8","#f58231",
    "#911eb4","#42d4f4","#f032e6","#bfef45","#fabed4","#469990",
]


# ── CSV helpers ───────────────────────────────────────────────
def save_mutation_csv(muts):
    df   = pd.DataFrame(muts)
    path = BASE_PATH + "mutations.csv"
    df.to_csv(path, index=False)
    print("Mutation CSV saved:", path)

def save_ddg_csv(ddg_results):
    df   = pd.DataFrame(ddg_results)
    path = BASE_PATH + "ddg_results.csv"
    df.to_csv(path, index=False)
    print("ΔΔG CSV saved:", path)

def save_epitope_csv(epitope_analysis: dict):
    rows = []
    for gene, data in epitope_analysis.items():
        if "error" in data:
            continue
        for src in ["bcell_parker","bcell_kolaskar","mhci","mhcii"]:
            for ep in data.get(src, []):
                ep["gene"]   = gene
                ep["source"] = src
                rows.append(ep)
    df   = pd.DataFrame(rows)
    path = BASE_PATH + "epitopes.csv"
    df.to_csv(path, index=False)
    print("Epitope CSV saved:", path)


# ── 1. Mutation frequency bar chart (annotated with genes) ────
def mutation_frequency_graph(muts):
    if not muts:
        print("No mutations to plot.")
        return

    positions = [m["position"] for m in muts]
    freq      = Counter(positions)

    fig, axes = plt.subplots(
        2, 1, figsize=(14, 7),
        gridspec_kw={"height_ratios": [6, 1]},
        sharex=True
    )

    # ── top: bar chart ─────────────────────────────────────────
    ax = axes[0]
    ax.bar(list(freq.keys()), list(freq.values()), width=20, color="#4363d8")
    ax.set_ylabel("Mutation Frequency")
    ax.set_title("Mutation Frequency Across RSV Genome")
    ax.grid(axis='y', linestyle='--', alpha=0.5)

    # ── bottom: gene annotation strip ─────────────────────────
    ax2 = axes[1]
    ax2.set_ylim(0, 1)
    ax2.set_yticks([])
    ax2.set_xlabel("Genome Position (nt)")

    for idx, (gene, (s, e)) in enumerate(RSV_GENE_MAP.items()):
        color = GENE_COLORS[idx % len(GENE_COLORS)]
        ax2.barh(0.5, e - s, left=s, height=0.6, color=color, alpha=0.85)
        mid = (s + e) / 2
        ax2.text(mid, 0.5, gene, ha='center', va='center',
                 fontsize=6, fontweight='bold', color='white')

    plt.tight_layout()
    path = BASE_PATH + "mutation_frequency.png"
    plt.savefig(path, dpi=150)
    plt.show()
    print("Mutation frequency graph saved:", path)


# ── 2. Mutation confidence heatmap ────────────────────────────
def mutation_heatmap(muts):
    if not muts:
        print("No mutations for heatmap.")
        return

    df = pd.DataFrame(muts)

    plt.figure(figsize=(14, 2))
    sns.heatmap(
        [df["confidence"].values],
        cmap="Reds", cbar=True,
        xticklabels=False, yticklabels=False
    )
    plt.title("Mutation Confidence Heatmap (each column = one mutation)")
    path = BASE_PATH + "mutation_heatmap.png"
    plt.savefig(path, dpi=150)
    plt.show()
    print("Heatmap saved:", path)


# ── 3. Per-gene mutation count bar chart ─────────────────────
def gene_mutation_chart(muts):
    if not muts:
        print("No mutations for gene chart.")
        return

    gene_counts = Counter([m.get("gene", "Unknown") for m in muts])
    genes  = list(gene_counts.keys())
    counts = list(gene_counts.values())

    plt.figure(figsize=(12, 5))
    bars = plt.bar(genes, counts, color=GENE_COLORS[:len(genes)], edgecolor='black')
    plt.xlabel("RSV Gene")
    plt.ylabel("Number of Predicted Mutations")
    plt.title("Predicted Mutations per RSV Gene")
    plt.xticks(rotation=30)

    for bar, count in zip(bars, counts):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(count), ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    path = BASE_PATH + "gene_mutation_chart.png"
    plt.savefig(path, dpi=150)
    plt.show()
    print("Gene mutation chart saved:", path)


# ── 4. ΔΔG per-gene heatmap ───────────────────────────────────
def ddg_gene_heatmap(ddg_results: list, gene_protein_changes: dict):
    """
    gene_protein_changes: output of compare_gene_proteins()
    ddg_results: flat list from compute_ddg()
    """
    from ddg_prediction import compute_ddg

    gene_ddg = {}
    for gene, data in gene_protein_changes.items():
        if "error" in data:
            continue
        _, total = compute_ddg(data.get("aa_changes", []))
        gene_ddg[gene] = total

    if not gene_ddg:
        print("No per-gene ΔΔG data to plot.")
        return

    genes  = list(gene_ddg.keys())
    values = [gene_ddg[g] for g in genes]

    plt.figure(figsize=(12, 2))
    sns.heatmap(
        [values],
        annot=True, fmt=".2f",
        xticklabels=genes,
        yticklabels=["ΔΔG"],
        cmap="RdYlGn_r",
        linewidths=0.5,
    )
    plt.title("Cumulative ΔΔG per RSV Gene")
    plt.tight_layout()
    path = BASE_PATH + "ddg_gene_heatmap.png"
    plt.savefig(path, dpi=150)
    plt.show()
    print("ΔΔG gene heatmap saved:", path)


# ── 5. Epitope map ─────────────────────────────────────────────
def epitope_map(epitope_analysis: dict):
    """
    Draw a horizontal bar for each gene protein, with
    B-cell / MHC-I / MHC-II epitope regions coloured.
    """
    fig, ax = plt.subplots(
        figsize=(14, len(RSV_GENE_MAP) * 0.7 + 1)
    )

    ep_colors = {
        "bcell_parker":   "#e6194b",
        "bcell_kolaskar": "#f58231",
        "mhci":           "#4363d8",
        "mhcii":          "#3cb44b",
    }

    y = 0
    yticks, ylabels = [], []

    for gene, data in epitope_analysis.items():
        if "error" in data:
            y += 1
            continue

        prot_len = data.get("length", 1)

        # grey background bar
        ax.barh(y, prot_len, left=0, height=0.5,
                color='#cccccc', alpha=0.4, edgecolor='black')

        for src, color in ep_colors.items():
            for ep in data.get(src, []):
                width = ep["end"] - ep["start"]
                ax.barh(y, width, left=ep["start"], height=0.5,
                        color=color, alpha=0.7)

        yticks.append(y)
        ylabels.append(gene)
        y += 1

    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels)
    ax.set_xlabel("Protein Position (aa)")
    ax.set_title("Predicted Epitope Map — All RSV Proteins")

    patches = [
        mpatches.Patch(color=c, label=s)
        for s, c in ep_colors.items()
    ]
    ax.legend(handles=patches, loc="lower right", fontsize=8)

    plt.tight_layout()
    path = BASE_PATH + "epitope_map.png"
    plt.savefig(path, dpi=150)
    plt.show()
    print("Epitope map saved:", path)


Overwriting visualization.py


In [ ]:
# ============================================================
# CELL 14 — run_test.py  (FULLY UPDATED)
# Runs: mutation prediction → per-gene protein analysis →
#       ΔΔG (full rule-based) → epitope prediction →
#       BERT embeddings → all visualizations
# ============================================================

%%writefile run_test.py
from data_preprocessing   import load_fasta
from mutation_prediction  import load_model_file, predict_mutations
from protein_analysis     import (
    dna_to_protein, apply_mutations,
    compare_proteins, compare_gene_proteins,
    get_all_gene_proteins,
)
from ddg_prediction       import compute_ddg, get_stability_suggestions
from epitope_prediction   import (
    run_full_epitope_analysis,
    check_mutation_in_epitope,
)
from bert_feature_extraction import get_bert_embeddings
from visualization        import (
    save_mutation_csv, save_ddg_csv, save_epitope_csv,
    mutation_frequency_graph, mutation_heatmap,
    gene_mutation_chart, ddg_gene_heatmap, epitope_map,
)
import pandas as pd

# ══════════════════════════════════════════════════════════════
# 1. LOAD DATA
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("LOADING SEQUENCES")
print("=" * 60)

test_seq = load_fasta("data/test_sequence.fasta")[0]
wild     = load_fasta("data/wild_type.fasta")[0]

print(f"Test sequence length : {len(test_seq)} nt")
print(f"Wild-type length     : {len(wild)} nt")

# ══════════════════════════════════════════════════════════════
# 2. LOAD MODEL + PREDICT MUTATIONS
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("MUTATION PREDICTION (LSTM)")
print("=" * 60)

model = load_model_file()
muts  = predict_mutations(model, test_seq)

print(f"Total predicted mutations : {len(muts)}")
print("\nFirst 10 mutations (with gene labels):")
for m in muts[:10]:
    print(f"  pos={m['position']:5d}  {m['actual']}→{m['predicted']}"
          f"  conf={m['confidence']:.3f}  gene={m['gene']}")

# Raw sequence differences
diff = sum(1 for i in range(min(len(test_seq), len(wild)))
           if test_seq[i] != wild[i])
print(f"\nRaw nucleotide differences vs wild-type: {diff}")

# ══════════════════════════════════════════════════════════════
# 3. WHOLE-GENOME PROTEIN ANALYSIS (ALL 11 GENES)
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PER-GENE PROTEIN ANALYSIS (NS1, NS2, N, P, M, SH, G, F, M2-1, M2-2, L)")
print("=" * 60)

mut_seq          = apply_mutations(wild, muts)
gene_comparisons = compare_gene_proteins(wild, mut_seq)

for gene, res in gene_comparisons.items():
    if "error" in res:
        print(f"  {gene}: ERROR — {res['error']}")
    else:
        print(f"  {gene:5s} | aa_changes={len(res['aa_changes']):4d} "
              f"| stop_changes={len(res['stop_changes'])}")

# ── flat lists for backward-compat sections below ────────────
p1 = dna_to_protein(wild)
p2 = dna_to_protein(mut_seq)
protein_changes, stop_codon_changes = compare_proteins(p1, p2)

print(f"\nTotal normal AA changes   : {len(protein_changes)}")
print(f"Total stop codon changes  : {len(stop_codon_changes)}")

print("\nFirst 5 stop codon mutations:")
for s in stop_codon_changes[:5]:
    print(" ", s)

# ══════════════════════════════════════════════════════════════
# 4. ΔΔG RULE-BASED (FULL)
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("ΔΔG ANALYSIS (FULL RULE-BASED)")
print("=" * 60)

ddg_results, total_ddg = compute_ddg(protein_changes)

print(f"Total ΔΔG : {total_ddg:.4f} kcal/mol (proxy)")
print("\nFirst 10 ΔΔG changes:")
for d in ddg_results[:10]:
    print(f"  pos={d['position']:4d}  {d['wild']}→{d['mutated']}"
          f"  ΔΔG={d['ddg']:+.3f}  {d['interpretation']}")

stability_suggestions = get_stability_suggestions(ddg_results)
print(f"\nStability improvement suggestions: {len(stability_suggestions)}")
for s in stability_suggestions[:5]:
    print(f"  pos={s['Position']}  {s['Current Mut']}→{s['Suggested AA']}: {s['Reason']}")

# ══════════════════════════════════════════════════════════════
# 5. BERT EMBEDDINGS (sequence-level, for NS1 & NS2 demo)
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BERT EMBEDDINGS (DNABERT-6) — NS1 + NS2 demo")
print("=" * 60)

from protein_analysis import extract_gene_sequence
ns1_cds = extract_gene_sequence(wild, "NS1")
ns2_cds = extract_gene_sequence(wild, "NS2")

ns_embeds = get_bert_embeddings([ns1_cds, ns2_cds], batch_size=2)
print(f"BERT embedding shape: {ns_embeds.shape}")    # (2, 768)
print("NS1 embedding vector (first 10 dims):", ns_embeds[0][:10])
print("NS2 embedding vector (first 10 dims):", ns_embeds[1][:10])

# ══════════════════════════════════════════════════════════════
# 6. EPITOPE PREDICTION (B-cell + MHC-I + MHC-II)
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("EPITOPE PREDICTION (ALL GENES)")
print("=" * 60)

# Set run_mhc=False to skip IEDB API if no internet
epitope_analysis = run_full_epitope_analysis(wild, run_mhc=True)

epitope_impact = check_mutation_in_epitope(muts, epitope_analysis)
print(f"\nMutations overlapping epitope regions: {len(epitope_impact)}")
for e in epitope_impact[:5]:
    print(f"  pos={e['position']}  gene={e['gene']}  "
          f"impact={e['epitope_disruption']}  "
          f"epitopes={len(e['overlapping_epitopes'])}")

# ══════════════════════════════════════════════════════════════
# 7. SAVE CSV FILES
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("SAVING CSV FILES")
print("=" * 60)

save_mutation_csv(muts)
save_ddg_csv(ddg_results)
save_epitope_csv(epitope_analysis)

pd.DataFrame(stability_suggestions).to_csv(
    "/content/drive/MyDrive/RSV_Project/stability_suggestions.csv", index=False
)
print("Stability suggestions CSV saved.")

pd.DataFrame(epitope_impact).to_csv(
    "/content/drive/MyDrive/RSV_Project/epitope_disruptions.csv", index=False
)
print("Epitope disruptions CSV saved.")

# ══════════════════════════════════════════════════════════════
# 8. VISUALIZATIONS
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("GENERATING VISUALIZATIONS")
print("=" * 60)

mutation_frequency_graph(muts)
mutation_heatmap(muts)
gene_mutation_chart(muts)
ddg_gene_heatmap(ddg_results, gene_comparisons)
epitope_map(epitope_analysis)

print("\n✅ run_test.py complete.")


Overwriting run_test.py


In [ ]:
# ============================================================
# CELL 15 — gradio_app.py  (FULLY UPDATED)
# New tabs: Per-Gene Proteins, Epitope Analysis,
#           BERT Embeddings, Stability Suggestions
# ============================================================

%%writefile gradio_app.py
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter

from data_preprocessing      import load_fasta
from mutation_prediction     import load_model_file, predict_mutations
from protein_analysis        import (
    dna_to_protein, apply_mutations,
    compare_proteins, compare_gene_proteins,
)
from ddg_prediction          import (
    compute_ddg, get_stability_suggestions,
)
from epitope_prediction      import (
    run_full_epitope_analysis,
    check_mutation_in_epitope,
)
from bert_feature_extraction import get_bert_embeddings
from visualization           import (
    save_mutation_csv, save_ddg_csv, save_epitope_csv,
)

BASE_PATH = "/content/drive/MyDrive/RSV_Project/"

RSV_GENE_MAP = {
    "NS1":(99,518),"NS2":(628,1002),"N":(1140,2315),
    "P":(2347,3072),"M":(3255,4025),"SH":(4295,4489),
    "G":(4681,5646),"F":(5726,7450),"M2-1":(7669,8253),
    "M2-2":(8228,8494),"L":(8561,15058),
}
GENE_COLORS=[
    "#e6194b","#3cb44b","#ffe119","#4363d8","#f58231",
    "#911eb4","#42d4f4","#f032e6","#bfef45","#fabed4","#469990",
]

# ── load model once ───────────────────────────────────────────
model = load_model_file()

# ══════════════════════════════════════════════════════════════
# MAIN ANALYSIS FUNCTION
# ══════════════════════════════════════════════════════════════
def analyze_fasta(file, run_mhc_api):

    # ── load sequences ─────────────────────────────────────────
    test_seq = load_fasta(file.name)[0]
    wild     = load_fasta("data/wild_type.fasta")[0]

    # ── mutation prediction ────────────────────────────────────
    muts    = predict_mutations(model, test_seq)
    mut_seq = apply_mutations(wild, muts)
    mut_df  = pd.DataFrame(muts)

    if len(mut_df) > 0:
        def sev(c):
            return "HIGH" if c > 0.90 else ("MEDIUM" if c > 0.75 else "LOW")
        mut_df["severity"] = mut_df["confidence"].apply(sev)

    # ── genome stats ───────────────────────────────────────────
    gc_content      = (test_seq.count("G") + test_seq.count("C")) / len(test_seq)
    mutation_density = len(muts) / len(test_seq)
    diff            = sum(1 for i in range(min(len(test_seq),len(wild)))
                          if test_seq[i] != wild[i])

    if len(muts) < 10:
        strain = "Mild Variant"
    elif len(muts) < 25:
        strain = "Moderately Mutated Variant"
    else:
        strain = "Highly Mutated Variant"

    # ── per-gene protein analysis ──────────────────────────────
    gene_comparisons = compare_gene_proteins(wild, mut_seq)
    gene_rows = []
    for gene, res in gene_comparisons.items():
        if "error" in res:
            continue
        gene_rows.append({
            "Gene":          gene,
            "AA Changes":    len(res.get("aa_changes", [])),
            "Stop Mutations":len(res.get("stop_changes", [])),
        })
    gene_df = pd.DataFrame(gene_rows)

    # ── flat protein comparison ─────────────────────────────────
    p1 = dna_to_protein(wild)
    p2 = dna_to_protein(mut_seq)
    protein_changes, stop_codon_changes = compare_proteins(p1, p2)

    stop_df = pd.DataFrame(stop_codon_changes)

    # ── ΔΔG ────────────────────────────────────────────────────
    ddg_results, total_ddg = compute_ddg(protein_changes)
    ddg_df  = pd.DataFrame(ddg_results)
    sug_df  = pd.DataFrame(get_stability_suggestions(ddg_results))

    ddg_df.to_csv(BASE_PATH + "ddg_results.csv", index=False)
    mut_df.to_csv(BASE_PATH + "mutations.csv",   index=False)

    # ── BERT embeddings (NS1 + NS2 representative) ────────────
    from protein_analysis import extract_gene_sequence
    ns1 = extract_gene_sequence(wild, "NS1")
    ns2 = extract_gene_sequence(wild, "NS2")
    bert_embeds = get_bert_embeddings([ns1, ns2], batch_size=2)
    bert_df = pd.DataFrame({
        "Gene":  ["NS1", "NS2"],
        "Embed_dim": [bert_embeds.shape[1]] * 2,
        "First_5_dims": [
            str(bert_embeds[0][:5].round(4).tolist()),
            str(bert_embeds[1][:5].round(4).tolist()),
        ]
    })

    # ── epitope prediction ─────────────────────────────────────
    epitope_analysis = run_full_epitope_analysis(
        wild, run_mhc=bool(run_mhc_api)
    )
    epitope_impact = check_mutation_in_epitope(muts, epitope_analysis)

    # flat epitope table
    ep_rows = []
    for gene, data in epitope_analysis.items():
        if "error" in data:
            continue
        for src in ["bcell_parker","bcell_kolaskar","mhci","mhcii"]:
            for ep in data.get(src, []):
                ep_rows.append({
                    "Gene":    gene,
                    "Type":    src,
                    "Start":   ep["start"],
                    "End":     ep["end"],
                    "Peptide": ep.get("peptide",""),
                    "Score":   ep.get("score", ep.get("ic50_nM",
                               ep.get("percentile",""))),
                })
    ep_df     = pd.DataFrame(ep_rows)
    impact_df = pd.DataFrame(epitope_impact)

    save_epitope_csv(epitope_analysis)

    # ── top 5 mutations ────────────────────────────────────────
    top_mut = (
        mut_df.sort_values("confidence", ascending=False).head(5)
        if len(mut_df) > 0
        else pd.DataFrame()
    )

    # ════════════════════════════════════════════════════════
    # PLOTS
    # ════════════════════════════════════════════════════════

    # ── mutation frequency + gene strip ───────────────────────
    positions = [m["position"] for m in muts]
    freq      = Counter(positions)

    fig_freq, axes = plt.subplots(
        2, 1, figsize=(12, 6),
        gridspec_kw={"height_ratios": [5, 1]}, sharex=True
    )
    axes[0].bar(list(freq.keys()), list(freq.values()), width=20, color="#4363d8")
    axes[0].set_ylabel("Frequency")
    axes[0].set_title("Mutation Frequency Distribution")

    for idx,(gene,(s,e)) in enumerate(RSV_GENE_MAP.items()):
        axes[1].barh(0.5, e-s, left=s, height=0.6,
                     color=GENE_COLORS[idx], alpha=0.85)
        axes[1].text((s+e)/2, 0.5, gene, ha='center', va='center',
                     fontsize=5, fontweight='bold', color='white')
    axes[1].set_yticks([])
    axes[1].set_xlabel("Genome Position (nt)")
    plt.tight_layout()

    # ── confidence heatmap ─────────────────────────────────────
    fig_conf, ax2 = plt.subplots(figsize=(12, 2))
    if len(mut_df) > 0:
        sns.heatmap([mut_df["confidence"].values],
                    cmap="Reds", cbar=True, ax=ax2,
                    xticklabels=False, yticklabels=False)
    ax2.set_title("Mutation Confidence Heatmap")
    plt.tight_layout()

    # ── gene mutation bar ──────────────────────────────────────
    fig_gene, ax3 = plt.subplots(figsize=(12, 4))
    if len(mut_df) > 0:
        gene_counts = mut_df["gene"].value_counts()
        ax3.bar(gene_counts.index, gene_counts.values,
                color=GENE_COLORS[:len(gene_counts)], edgecolor='black')
        ax3.set_xlabel("RSV Gene")
        ax3.set_ylabel("Mutation Count")
        ax3.set_title("Mutations per RSV Gene")
        ax3.tick_params(axis='x', rotation=30)
    plt.tight_layout()

    # ── epitope map ────────────────────────────────────────────
    ep_colors_map = {
        "bcell_parker":"#e6194b","bcell_kolaskar":"#f58231",
        "mhci":"#4363d8","mhcii":"#3cb44b",
    }
    fig_ep, ax_ep = plt.subplots(
        figsize=(13, len(RSV_GENE_MAP) * 0.65 + 1)
    )
    y = 0
    yticks, ylabels = [], []
    for gene, data in epitope_analysis.items():
        if "error" in data:
            y += 1; continue
        plen = data.get("length", 1)
        ax_ep.barh(y, plen, left=0, height=0.5,
                   color='#cccccc', alpha=0.4, edgecolor='black')
        for src, col in ep_colors_map.items():
            for ep in data.get(src, []):
                ax_ep.barh(y, ep["end"]-ep["start"], left=ep["start"],
                           height=0.5, color=col, alpha=0.7)
        yticks.append(y); ylabels.append(gene); y += 1

    ax_ep.set_yticks(yticks); ax_ep.set_yticklabels(ylabels)
    ax_ep.set_xlabel("Protein Position (aa)")
    ax_ep.set_title("Predicted Epitope Map — All RSV Proteins")
    patches = [mpatches.Patch(color=c, label=s)
               for s, c in ep_colors_map.items()]
    ax_ep.legend(handles=patches, loc="lower right", fontsize=8)
    plt.tight_layout()

    # ── load training plots ────────────────────────────────────
    accuracy_img     = BASE_PATH + "accuracy_plot.png"
    loss_img         = BASE_PATH + "loss_plot.png"
    confusion_img    = BASE_PATH + "confusion_matrix.png"

    # ── summary markdown ──────────────────────────────────────
    summary = f"""
# 🧬 RSV Mutation Analysis Report

---
## 📊 Deep Learning Model Performance
| Metric | Score |
|---|---|
| Validation Accuracy | 97.49% |
| Precision | 98.92% |
| Recall | 98.92% |
| F1 Score | 98.91% |

---
## 🧬 Genome Statistics
| Parameter | Value |
|---|---|
| Genome Length | {len(test_seq)} nt |
| GC Content | {round(gc_content, 4)} |
| Predicted Mutations | {len(muts)} |
| Mutation Density | {round(mutation_density, 6)} |
| Raw Sequence Differences | {diff} |
| Variant Classification | **{strain}** |

---
## 🧪 Protein Stability (ΔΔG)
| Parameter | Value |
|---|---|
| Total AA Changes | {len(protein_changes)} |
| Stop Codon Mutations | {len(stop_codon_changes)} |
| Total ΔΔG | {round(total_ddg, 4)} kcal/mol (proxy) |
| Stabilisation Suggestions | {len(get_stability_suggestions(ddg_results))} |

---
## 🔬 BERT Feature Extraction
DNABERT-6 embeddings extracted for NS1 and NS2 genes.
Embedding dimension: **768** per sequence.

---
## 🧫 Epitope Summary
| Type | Count |
|---|---|
| B-cell (Parker) | {sum(len(v.get('bcell_parker',[])) for v in epitope_analysis.values() if 'error' not in v)} |
| B-cell (Kolaskar) | {sum(len(v.get('bcell_kolaskar',[])) for v in epitope_analysis.values() if 'error' not in v)} |
| MHC-I binders | {sum(len(v.get('mhci',[])) for v in epitope_analysis.values() if 'error' not in v)} |
| MHC-II binders | {sum(len(v.get('mhcii',[])) for v in epitope_analysis.values() if 'error' not in v)} |
| Mutations in epitopes | {len(epitope_impact)} |
"""

    return (
        summary,
        mut_df, top_mut,
        gene_df,
        stop_df, ddg_df, sug_df,
        bert_df,
        ep_df, impact_df,
        fig_freq, fig_conf, fig_gene, fig_ep,
        accuracy_img, loss_img, confusion_img,
        BASE_PATH + "mutations.csv",
        BASE_PATH + "ddg_results.csv",
        BASE_PATH + "epitopes.csv",
    )


# ══════════════════════════════════════════════════════════════
# GRADIO UI
# ══════════════════════════════════════════════════════════════
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
# 🧬 RSV Mutation Prediction & Protein Stability Analysis
### LSTM · DNABERT-6 · Rule-Based ΔΔG · Epitope Mapping
""")

    with gr.Row():
        file_input   = gr.File(label="Upload Test FASTA File")
        run_mhc_api  = gr.Checkbox(
            label="Run MHC-I / MHC-II via IEDB API (requires internet)",
            value=True
        )

    analyze_btn = gr.Button("🔬 Run Full Analysis", variant="primary")

    # ── Overview ───────────────────────────────────────────────
    with gr.Tab("📊 Overview"):
        with gr.Row():
            gr.Number(label="Val Accuracy (%)", value=97.49)
            gr.Number(label="Precision (%)",    value=98.92)
            gr.Number(label="Recall (%)",       value=98.92)
            gr.Number(label="F1 Score (%)",     value=98.91)
        summary_out = gr.Markdown()

    # ── Mutations ──────────────────────────────────────────────
    with gr.Tab("🔴 Mutation Analysis"):
        mut_out  = gr.Dataframe(label="All Predicted Mutations")
        top_out  = gr.Dataframe(label="Top 5 High-Confidence Mutations")
        mut_csv  = gr.File(label="Download Mutations CSV")

    # ── Per-Gene Proteins ──────────────────────────────────────
    with gr.Tab("🧬 Per-Gene Proteins"):
        gr.Markdown("### AA Changes and Stop Codon Mutations per RSV Gene")
        gene_out = gr.Dataframe(label="Gene-Level Protein Changes")

    # ── Protein Stability ──────────────────────────────────────
    with gr.Tab("🧪 Protein Stability (ΔΔG)"):
        stop_out = gr.Dataframe(label="Stop Codon Mutations")
        ddg_out  = gr.Dataframe(label="ΔΔG Results (Rule-Based)")
        ddg_csv  = gr.File(label="Download ΔΔG CSV")
        sug_out  = gr.Dataframe(label="Stabilisation Suggestions")

    # ── BERT Embeddings ────────────────────────────────────────
    with gr.Tab("🤖 BERT Embeddings"):
        gr.Markdown("""
### DNABERT-6 Feature Extraction
Sequences are tokenised into overlapping 6-mers and fed to a
pre-trained DNABERT transformer. The [CLS] token embedding
(768-dim) provides a rich contextual representation for each gene.
""")
        bert_out = gr.Dataframe(label="BERT Embeddings Summary (NS1 & NS2)")

    # ── Epitope Analysis ───────────────────────────────────────
    with gr.Tab("🧫 Epitope Analysis"):
        gr.Markdown("""
### B-cell & T-cell Epitope Prediction
- **B-cell (Parker)** — hydrophilicity sliding-window score
- **B-cell (Kolaskar)** — antigenicity propensity scale
- **MHC-I** — NetMHCpan via IEDB REST (HLA-A*02:01, 9-mer)
- **MHC-II** — NetMHCIIpan via IEDB REST (DRB1*01:01, 15-mer)
""")
        ep_out     = gr.Dataframe(label="All Predicted Epitopes")
        impact_out = gr.Dataframe(label="Mutations Overlapping Epitope Regions ⚠")
        ep_csv     = gr.File(label="Download Epitope CSV")

    # ── Visualizations ─────────────────────────────────────────
    with gr.Tab("📈 Visualizations"):
        freq_plot  = gr.Plot(label="Mutation Frequency + Gene Map")
        conf_plot  = gr.Plot(label="Confidence Heatmap")
        gene_plot  = gr.Plot(label="Mutations per Gene")
        ep_plot    = gr.Plot(label="Epitope Map")
        acc_img    = gr.Image(label="Training Accuracy")
        loss_img_o = gr.Image(label="Training Loss")
        cm_img     = gr.Image(label="Confusion Matrix")

    # ── button wiring ──────────────────────────────────────────
    analyze_btn.click(
        fn=analyze_fasta,
        inputs=[file_input, run_mhc_api],
        outputs=[
            summary_out,
            mut_out, top_out,
            gene_out,
            stop_out, ddg_out, sug_out,
            bert_out,
            ep_out, impact_out,
            freq_plot, conf_plot, gene_plot, ep_plot,
            acc_img, loss_img_o, cm_img,
            mut_csv, ddg_csv, ep_csv,
        ]
    )

demo.launch(share=True)


Overwriting gradio_app.py


In [ ]:
# ============================================================
# CELL 7 — main.py  (UNCHANGED — trains the LSTM)
# ============================================================

%%writefile main.py
from data_preprocessing  import load_fasta, preprocess
from redundancy_removal  import remove_redundant
from feature_extraction  import create_dataset
from model               import train_model
from evaluation          import evaluate_model
from tensorflow.keras.models import save_model

seqs = load_fasta("data/sequence.fasta", limit=6000)
seqs = preprocess(seqs)

print("Removing redundancy...")
seqs, removed = remove_redundant(seqs)
print("Final sequences after redundancy removal:", len(seqs))

X, y = create_dataset(seqs)

model = train_model(X, y)
evaluate_model(model, X[:5000], y[:5000])

model.save("model.keras")
print("✅ Model saved.")


Overwriting main.py


In [ ]:
!python main.py

2026-06-03 09:00:53.572233: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Initial: 6000
After length filter: 5756
After removing N: 5682
After removing duplicates: 5467
Removing redundancy...
Final sequences after redundancy removal: 408
Final sequences after redundancy removal: 408
2026-06-03 09:02:25.831792: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1780477345.833317    4975 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Epoch 1/25
2026-06-03 09:02:31.37

In [ ]:
!python run_test.py

2026-06-05 06:10:49.589969: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
LOADING SEQUENCES
Test sequence length : 15231 nt
Wild-type length     : 15191 nt

MUTATION PREDICTION (LSTM)
2026-06-05 06:11:14.024607: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1780639874.026118    2365 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
2026-06-05 06:11:16.885595: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900
Total predicted

In [ ]:
!python gradio_app.py

2026-06-05 06:24:17.462846: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-05 06:24:27.276116: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1780640667.277580    5755 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
/content/drive/MyDrive/RSV_Project/gradio_app.py:298: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes